# 🧪 Notebook 02: ADMET Triage, Chemical-Space PCA, and Drug Similarity (Features E3, E9, E10)
Computes Lipinski/Veber permeability zones, runs Morgan Fingerprint Tanimoto similarity against Positive Controls, and projects chemical space via PCA.


In [ ]:
!pip install -q rdkit pandas matplotlib scikit-learn
import pandas as pd
import numpy as np
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

df = pd.read_csv('../data/durva_compounds_v1.csv')
df[['compound_name', 'mol_weight', 'logP', 'TPSA', 'lipinski_pass', 'gi_absorption']]


In [ ]:
# Feature E9 & E10: Morgan Fingerprints + Tanimoto Similarity to Celecoxib + 2D PCA Chemical Space
mols = [Chem.MolFromSmiles(s) for s in df['smiles']]
fps = [AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=1024) for m in mols]

# Similarity against Celecoxib (index -2)
celecoxib_fp = fps[-2]
df['similarity_to_celecoxib'] = [round(DataStructs.TanimotoSimilarity(fp, celecoxib_fp), 3) for fp in fps]

# PCA of Chemical Space
fp_matrix = np.array([list(fp) for fp in fps])
pca = PCA(n_components=2)
coords = pca.fit_transform(fp_matrix)
df['PC1'], df['PC2'] = coords[:, 0], coords[:, 1]
df[['compound_name', 'similarity_to_celecoxib', 'PC1', 'PC2']]
